#**LAB 4 (07 Feb- 08 Feb 2025)**
#TOPICS COVERED : Neural Networks

# There are 2 graded questions marked `# GRADED` in the following section. .

Download datasets for this lab from this link: https://drive.google.com/drive/folders/144TVoP81peoe5yY4vVH6WBXGVsCHDoRK?usp=sharing

In [ ]:
import pandas as pd
import numpy as np
import torch

Loading the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
t_indep=pd.read_csv('/content/drive/MyDrive/DL_LAB/LAB4/indep.csv')
t_indep=t_indep.drop(t_indep.columns[[0]],axis=1)
t_dep=pd.read_csv('/content/drive/MyDrive/DL_LAB/LAB4/dep.csv')
t_dep=t_dep.drop(t_dep.columns[[0]],axis=1)

In [ ]:
print(t_indep)

      Age  SibSp  Parch   LogFare  Sex_male  Sex_female  Pclass_1  Pclass_2  \
0    22.0      1      0  2.110213         1           0         0         0   
1    38.0      1      0  4.280593         0           1         1         0   
2    26.0      0      0  2.188856         0           1         0         0   
3    35.0      1      0  3.990834         0           1         1         0   
4    35.0      0      0  2.202765         1           0         0         0   
..    ...    ...    ...       ...       ...         ...       ...       ...   
886  27.0      0      0  2.639057         1           0         0         1   
887  19.0      0      0  3.433987         0           1         1         0   
888  24.0      1      2  3.196630         0           1         0         0   
889  26.0      0      0  3.433987         1           0         1         0   
890  32.0      0      0  2.169054         1           0         0         0   

     Pclass_3  Embarked_C  Embarked_Q  Embarked_S  

In [ ]:
print(t_dep)

     Survived
0           0
1           1
2           1
3           1
4           0
..        ...
886         0
887         1
888         0
889         1
890         0

[891 rows x 1 columns]


In [ ]:
print(t_dep)

     Survived
0           0
1           1
2           1
3           1
4           0
..        ...
886         0
887         1
888         0
889         1
890         0

[891 rows x 1 columns]


So PyTorch tensors allow us to use gradients and make life easier, so we'll convert these dataframe values into those


In [ ]:
t_dep=torch.tensor(t_dep.values,dtype=torch.float)
t_indep=torch.tensor(t_indep.values,dtype=torch.float)

In [ ]:
print(t_indep.size())

torch.Size([891, 12])


In [ ]:
print(t_dep.size())

torch.Size([891, 1])


Setting up a Linear Model

In [ ]:
# a*x1 + b*x2 + c*x3 + d*x4 .....
# the coeffs below are a,b,c,d,.....

In [ ]:
t_indep.size()

torch.Size([891, 12])

In [ ]:
torch.manual_seed(42)

n_coeff=t_indep.size()[1]
coeffs=torch.rand(n_coeff)-0.5 #random values in range -0.5 to 0.5
coeffs

tensor([ 0.3823,  0.4150, -0.1171,  0.4593, -0.1096,  0.1009, -0.2434,  0.2936,
         0.4408, -0.3668,  0.4346,  0.0936])

Normalizing values in each column

This is done to prevent any one column dominating the prediction results, since a linear model is row*coeffs, very high initial values in some columns will lead to some columns dominating the final answer which we don't want

In [ ]:
#finds the maximum value in each column (feature) of the t_indep tensor.
vals,indices = t_indep.max(dim=0) #dim=0: Specifies that the operation should be performed along the columns.
t_indep = t_indep / vals  #v cool line of code, think about why


In [ ]:
vals


tensor([80.0000,  8.0000,  6.0000,  6.2409,  1.0000,  1.0000,  1.0000,  1.0000,
         1.0000,  1.0000,  1.0000,  1.0000])

In [ ]:
indices

tensor([630, 159, 678, 258,   0,   1,   1,   9,   0,   1,   5,   0])

In [ ]:
# t_indep(891*12) vals(1*12) matrix
# for i in range(12):
#   t_indep[i] = t_indep[i]/vals[i]

In [ ]:
#Each feature in a row of t_indep is multiplied by its corresponding coefficient in coeffs.

#The sum represents the weighted sum of the features for each data point, which
#is the predicted value for that data point.

"""This line computes the predicted values (preds) for each data point in your dataset
by taking the dot product of the feature matrix (t_indep) and the coefficient vector (coeffs).
For example, if t_indep has 5 rows (data points) and 12 columns (features),
and coeffs has 12 elements (one for each feature),
the result will be a tensor preds with 5 elements,
each representing the predicted value for a corresponding data point."""

'This line computes the predicted values (preds) for each data point in your dataset\nby taking the dot product of the feature matrix (t_indep) and the coefficient vector (coeffs).\nFor example, if t_indep has 5 rows (data points) and 12 columns (features),\nand coeffs has 12 elements (one for each feature),\nthe result will be a tensor preds with 5 elements,\neach representing the predicted value for a corresponding data point.'

In [ ]:
preds=(t_indep*coeffs).sum(axis=1)

In [ ]:
preds[0:10]



tensor([0.7371, 0.0391, 0.9206, 0.4639, 0.7542, 1.0459, 0.2906, 0.7982, 0.9089,
        0.3994])

In [ ]:

# Alternate way to do the same thing
torch.matmul(t_indep,coeffs)[:10]

tensor([0.7371, 0.0391, 0.9206, 0.4639, 0.7542, 1.0459, 0.2906, 0.7982, 0.9089,
        0.3994])

In [ ]:
# Let's see the above operations more clearly

print('First 5 rows of t_indep')
print(t_indep[:5,:].numpy()) #select the first 5 rows and all columns
print('\n vals')
print(vals)
print('\n indices')
print(indices)

First 5 rows of t_indep
[[0.275      0.125      0.         0.3381255  1.         0.
  0.         0.         1.         0.         0.         1.        ]
 [0.475      0.125      0.         0.6858916  0.         1.
  1.         0.         0.         1.         0.         0.        ]
 [0.325      0.         0.         0.35072672 0.         1.
  0.         0.         1.         0.         0.         1.        ]
 [0.4375     0.125      0.         0.63946277 0.         1.
  1.         0.         0.         0.         0.         1.        ]
 [0.4375     0.         0.         0.35295528 1.         0.
  0.         0.         1.         0.         0.         1.        ]]

 vals
tensor([80.0000,  8.0000,  6.0000,  6.2409,  1.0000,  1.0000,  1.0000,  1.0000,
         1.0000,  1.0000,  1.0000,  1.0000])

 indices
tensor([630, 159, 678, 258,   0,   1,   1,   9,   0,   1,   5,   0])


In [ ]:
t_indep = t_indep / vals  #v cool line of code, think about why

In [ ]:
print('First 5 rows of t_indep')
# print(t_indep[:5,:].numpy())
print(t_indep[:5,:])

First 5 rows of t_indep
tensor([[0.0034, 0.0156, 0.0000, 0.0542, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 1.0000],
        [0.0059, 0.0156, 0.0000, 0.1099, 0.0000, 1.0000, 1.0000, 0.0000, 0.0000,
         1.0000, 0.0000, 0.0000],
        [0.0041, 0.0000, 0.0000, 0.0562, 0.0000, 1.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 1.0000],
        [0.0055, 0.0156, 0.0000, 0.1025, 0.0000, 1.0000, 1.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 1.0000],
        [0.0055, 0.0000, 0.0000, 0.0566, 1.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 1.0000]])


In [ ]:
preds=(t_indep*coeffs)

In [ ]:
# ax11 bx21 cx31 dx41 ...... 12 columns towards the right
# ax12 bx22 cx32 dx42
# ...... 891 rows this way downwards

In [ ]:
# print(preds)
print(preds.shape)
print(t_indep.shape)
print(coeffs.shape)  # initial coeffs..

torch.Size([891, 12])
torch.Size([891, 12])
torch.Size([12])


In [ ]:
preds=preds.sum(axis=1)

In [ ]:
print(preds.shape)

torch.Size([891])


In [ ]:

preds[0:10]

tensor([ 0.4575, -0.4501,  0.6626,  0.0067,  0.4529,  0.7937, -0.2094,  0.4776,
         0.6598,  0.0756])

In [ ]:
loss=torch.abs(preds-t_dep).mean()
loss

tensor(0.5414)



```
# This is formatted as code
```

Making functions for loss and predictions

In [ ]:
def pred(coeffs,indeps):
  return (indeps*coeffs).sum(axis=1)

def calc_loss(coeffs,indeps,deps):
  return torch.abs(pred(coeffs,indeps)-deps).mean() #calculates the mean absolute error between the predicted values and the actual values.

Doing gradient descent

In [ ]:
coeffs.requires_grad_() #enables gradients for coeffs tensor
 #It allows PyTorch to track operations on the coeffs tensor so that gradients
 #can be computed during backpropagation.
 #his is essential for training neural networks, as it allows the model to
 #update the coefficients based on the computed gradients during the optimization process.

tensor([ 0.3823,  0.4150, -0.1171,  0.4593, -0.1096,  0.1009, -0.2434,  0.2936,
         0.4408, -0.3668,  0.4346,  0.0936], requires_grad=True)

In [ ]:
loss=calc_loss(coeffs,t_indep,t_dep)
loss

tensor(0.5414, grad_fn=<MeanBackward0>)

In [ ]:
loss.backward() #calculates gradients

#erforms backpropagation to compute the gradients of the loss with respect
#to the model parameters (in this case, the coeffs tensor).

In [ ]:
coeffs.grad

tensor([-4.9219e-04,  5.8809e-04,  1.4738e-04, -8.9413e-03, -7.7758e-02,
         3.6141e-02, -1.8433e-01,  3.4146e-02,  1.0857e-01, -1.3461e-01,
         4.7648e-02,  4.5347e-02])

Each time you call loss.backwards, newly calculated gradients are accumulated (or added to current gradients)

We use tensor.grad.zero_() to make the gradients zero after each step

In [ ]:
coeffs.grad.zero_() # making coeffs zero for the coeffs tensor
loss=calc_loss(coeffs,t_indep,t_dep) # calculating the loss
loss.backward() # backpropagation step
# torch.no_grad means that we will not change the values of any gradients for any tensor inside this loop
with torch.no_grad():
  coeffs.sub_(coeffs.grad*0.1) # coeffs_new = coeffs_old - learning_rate * gradients
  coeffs.grad.zero_()
  print(calc_loss(coeffs,t_indep,t_dep))

tensor(0.5343)


Making functions for this

In [ ]:
def update_coeffs(coeffs,lr):
  coeffs.sub_(coeffs.grad*lr)
  coeffs.grad.zero_()

In [ ]:
def one_epoch(coeffs,lr):
  loss=calc_loss(coeffs,t_indep,t_dep)
  loss.backward()
  with torch.no_grad():
    update_coeffs(coeffs,lr)
    print(f"{loss:.3f}", end="; ")


In [ ]:
def init_coeffs():
  return (torch.rand(n_coeff)-0.5).requires_grad_()

In [ ]:
def train_model(epochs=30000, lr=0.01):
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): one_epoch(coeffs, lr=lr)
    return coeffs

In [ ]:
train_model(epochs=30)

0.522; 0.521; 0.520; 0.520; 0.519; 0.518; 0.518; 0.517; 0.516; 0.515; 0.515; 0.514; 0.513; 0.513; 0.512; 0.511; 0.511; 0.510; 0.509; 0.508; 0.508; 0.507; 0.506; 0.506; 0.505; 0.504; 0.503; 0.503; 0.502; 0.501; 

tensor([-0.4629,  0.1384,  0.2409, -0.2252, -0.2690, -0.3090,  0.5060,  0.3062,
         0.2689, -0.3826,  0.2043,  0.3120], requires_grad=True)

In [ ]:
print(coeffs.grad) #shows us that the model has pretty much converged

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


Building a full neural network

In [ ]:
import torch.nn.functional as F

def preds(coeffs, indeps):
    l1,l2,const = coeffs
    res = F.relu(indeps@l1) # '@' is an optimized matrix product in python
    res = res@l2 + const
    return torch.sigmoid(res)



In [ ]:
def calc_loss(coeffs,indeps,deps):
  return torch.abs(preds(coeffs,indeps)-deps).mean()

In [ ]:
def one_epoch(coeffs,lr):
  loss=calc_loss(coeffs,t_indep,t_dep)
  loss.backward()
  with torch.no_grad():
    update_coeffs(coeffs,lr)
    print(f"{loss:.3f}", end="; ")

In [ ]:
def train_model(epochs=300, lr=0.1):
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs): one_epoch(coeffs, lr=lr)
    return coeffs

In [ ]:
def init_coeffs(n_hidden=60):
    layer1 = (torch.rand(n_coeff, n_hidden)-0.5)/n_hidden
    layer2 = torch.rand(n_hidden, 1)-0.3
    const = torch.rand(1)[0]
    return layer1.requires_grad_(),layer2.requires_grad_(),const.requires_grad_()

In [ ]:
def update_coeffs(coeffs, lr=0.1):
    for layer in coeffs:
        layer.sub_(layer.grad * lr)
        layer.grad.zero_()

In [ ]:
coeffs=train_model()
# update_coeffs(l1)


0.524; 0.520; 0.518; 0.516; 0.515; 0.513; 0.511; 0.510; 0.508; 0.506; 0.504; 0.502; 0.500; 0.498; 0.496; 0.494; 0.492; 0.490; 0.488; 0.486; 0.483; 0.481; 0.479; 0.477; 0.475; 0.473; 0.471; 0.468; 0.466; 0.464; 0.462; 0.460; 0.457; 0.455; 0.453; 0.451; 0.448; 0.446; 0.444; 0.442; 0.439; 0.437; 0.435; 0.432; 0.430; 0.428; 0.426; 0.423; 0.421; 0.419; 0.416; 0.414; 0.412; 0.409; 0.407; 0.404; 0.402; 0.400; 0.397; 0.395; 0.392; 0.390; 0.388; 0.385; 0.383; 0.380; 0.378; 0.375; 0.373; 0.371; 0.368; 0.366; 0.363; 0.361; 0.359; 0.356; 0.354; 0.352; 0.349; 0.347; 0.345; 0.342; 0.340; 0.338; 0.336; 0.334; 0.332; 0.329; 0.327; 0.325; 0.323; 0.321; 0.319; 0.317; 0.315; 0.314; 0.312; 0.310; 0.308; 0.306; 0.305; 0.303; 0.301; 0.300; 0.298; 0.297; 0.295; 0.294; 0.292; 0.291; 0.290; 0.288; 0.287; 0.286; 0.284; 0.283; 0.282; 0.281; 0.279; 0.278; 0.277; 0.276; 0.275; 0.274; 0.273; 0.272; 0.271; 0.270; 0.269; 0.268; 0.267; 0.266; 0.265; 0.265; 0.264; 0.263; 0.262; 0.261; 0.261; 0.260; 0.259; 0.259; 0.258;

In [ ]:
def acc(coeffs): return (t_dep.bool()==(preds(coeffs,t_indep)>0.5)).float().mean()



In [ ]:
acc(coeffs)

tensor(0.6285)

loss.backwards() on a linear model

*   List item
*   List item


ax + by + cz + dw.....

S = ax + by + cz + dw.....

delta = partial derivative of S wrt a
a_new = a_old - lr * delta

# **Graded questions 1 and 2**


In [ ]:

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# Assuming t_indep and t_dep are already defined as in your original code

# Split the data into training and temporary sets (test + validation)
X_train, X_temp, y_train, y_temp = train_test_split(
    t_indep, t_dep, test_size=0.3, random_state=42
)

# Split the temporary set into test and validation sets
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Now you have:
# X_train, y_train: Training data
# X_val, y_val: Validation data
# X_test, y_test: Testing data

print("Training data shapes:", X_train.shape, y_train.shape)
print("Validation data shapes:", X_val.shape, y_val.shape)
print("Test data shapes:", X_test.shape, y_test.shape)


Training data shapes: torch.Size([623, 12]) torch.Size([623, 1])
Validation data shapes: torch.Size([134, 12]) torch.Size([134, 1])
Test data shapes: torch.Size([134, 12]) torch.Size([134, 1])


#1 Create a train/test/validation split from the dataset(80:10:10)

In [ ]:
import pandas as pd
import numpy as np
import torch
from google.colab import drive
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# Load the datasets (assuming they are in your Google Drive)
drive.mount('/content/drive')
t_indep = pd.read_csv('/content/drive/MyDrive/DL_LAB/LAB4/indep.csv')
t_indep = t_indep.drop(t_indep.columns[[0]], axis=1)
t_dep = pd.read_csv('/content/drive/MyDrive/DL_LAB/LAB4/dep.csv')
t_dep = t_dep.drop(t_dep.columns[[0]], axis=1)

# Convert to PyTorch tensors
t_dep = torch.tensor(t_dep.values, dtype=torch.float)
t_indep = torch.tensor(t_indep.values, dtype=torch.float)

# Split the data into train, validation, and test sets (80:10:10)
X_train, X_temp, y_train, y_temp = train_test_split(
    t_indep, t_dep, test_size=0.2, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Print the shapes of the resulting sets
print("Training data shapes:", X_train.shape, y_train.shape)
print("Validation data shapes:", X_val.shape, y_val.shape)
print("Test data shapes:", X_test.shape, y_test.shape)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training data shapes: torch.Size([712, 12]) torch.Size([712, 1])
Validation data shapes: torch.Size([89, 12]) torch.Size([89, 1])
Test data shapes: torch.Size([90, 12]) torch.Size([90, 1])


#2 Evaluate the testing split and accuracy
# Add one hidden layer to improve its accuracy ( lr=0.01 Epoch = 200 )

In [ ]:
import torch.nn.functional as F

def preds(coeffs, indeps):
    l1, l2, l3, const = coeffs  # Added l3 for the new hidden layer
    res = F.relu(indeps @ l1)  # First hidden layer
    res = F.relu(res @ l2)  # Second hidden layer
    res = res @ l3 + const    # Output layer
    return torch.sigmoid(res)

def calc_loss(coeffs, indeps, deps):
    return torch.abs(preds(coeffs, indeps) - deps).mean()

def one_epoch(coeffs, lr):
    loss = calc_loss(coeffs, t_indep, t_dep)
    loss.backward()
    with torch.no_grad():
        for layer in coeffs:
            layer.sub_(layer.grad * lr)
            layer.grad.zero_()
    print(f"{loss:.3f}", end="; ")  # Print loss for each epoch


def init_coeffs(n_hidden1=60, n_hidden2=30):  # Added n_hidden2
    layer1 = (torch.rand(n_coeff, n_hidden1) - 0.5) / n_hidden1
    layer2 = (torch.rand(n_hidden1, n_hidden2) - 0.5) / n_hidden2  # New hidden layer
    layer3 = torch.rand(n_hidden2, 1) - 0.3  # Output layer
    const = torch.rand(1)[0]
    return layer1.requires_grad_(), layer2.requires_grad_(), layer3.requires_grad_(), const.requires_grad_()

def train_model(epochs=200, lr=0.1):  # Changed epochs and lr
    torch.manual_seed(442)
    coeffs = init_coeffs()
    for i in range(epochs):
        one_epoch(coeffs, lr=lr)
    return coeffs

# Assuming n_coeff and t_indep, t_dep are defined
coeffs = train_model(epochs=200, lr=0.1)

def acc(coeffs):
    return (t_dep.bool()==(preds(coeffs,t_indep)>0.5)).float().mean()

acc(coeffs)

0.539; 0.537; 0.536; 0.535; 0.534; 0.534; 0.533; 0.532; 0.530; 0.529; 0.528; 0.526; 0.523; 0.521; 0.517; 0.513; 0.508; 0.501; 0.493; 0.483; 0.472; 0.460; 0.447; 0.435; 0.424; 0.415; 0.407; 0.401; 0.397; 0.393; 0.390; 0.388; 0.387; 0.385; 0.384; 0.383; 0.382; 0.382; 0.381; 0.381; 0.380; 0.380; 0.379; 0.379; 0.379; 0.379; 0.379; 0.378; 0.378; 0.378; 0.378; 0.378; 0.378; 0.378; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.377; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.376; 0.375; 0.375; 0.375; 0.375; 0.375;

tensor(0.6285)